In [36]:
import pandas as pd
import re
import requests
import ast
from tqdm import tqdm
import time

In [37]:
books_df = pd.read_csv('../data/raw/books.csv')
books_df.head()

,Unnamed: 0,index,authors,average_rating,best_book_id,book_id,books_count,description,genres,goodreads_book_id,...,ratings_3,ratings_4,ratings_5,ratings_count,small_image_url,title,work_id,work_ratings_count,work_text_reviews_count,authors_2
0,0,0,['Suzanne Collins'],4.34,2767052,1,272,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,"['young-adult', 'fiction', 'fantasy', 'science...",2767052,...,560092,1481305,2706317,4780653,https://images.gr-assets.com/books/1447303603s...,"The Hunger Games (The Hunger Games, #1)",2792775,4942365,155254,['Suzanne Collins']
1,1,1,"['J.K. Rowling', 'Mary GrandPré']",4.44,3,2,491,Harry Potter's life is miserable. His parents ...,"['fantasy', 'fiction', 'young-adult', 'classics']",3,...,455024,1156318,3011543,4602479,https://images.gr-assets.com/books/1474154022s...,Harry Potter and the Sorcerer's Stone (Harry P...,4640799,4800065,75867,"['J.K. Rowling', 'Mary GrandPré']"
2,2,2,['Stephenie Meyer'],3.57,41865,3,226,About three things I was absolutely positive. ...,"['young-adult', 'fantasy', 'romance', 'fiction...",41865,...,793319,875073,1355439,3866839,https://images.gr-assets.com/books/1361039443s...,"Twilight (Twilight, #1)",3212258,3916824,95009,['Stephenie Meyer']
3,3,3,['Harper Lee'],4.25,2657,4,487,The unforgettable novel of a childhood in a sl...,"['classics', 'fiction', 'historical-fiction', ...",2657,...,446835,1001952,1714267,3198671,https://images.gr-assets.com/books/1361975680s...,To Kill a Mockingbird,3275794,3340896,72586,['Harper Lee']
4,4,4,['F. Scott Fitzgerald'],3.89,4671,5,1356,Alternate Cover Edition ISBN: 0743273567 (ISBN...,"['classics', 'fiction', 'historical-fiction', ...",4671,...,606158,936012,947718,2683664,https://images.gr-assets.com/books/1490528560s...,The Great Gatsby,245494,2773745,51992,['F. Scott Fitzgerald']


In [38]:
columns_to_keep = ['book_id', 'title', 'authors', 'average_rating', 'description', 'genres', 'pages', 'original_publication_year']
cleaned_books_df = books_df[columns_to_keep]
cleaned_books_df.head()

,book_id,title,authors,average_rating,description,genres,pages,original_publication_year
0,1,"The Hunger Games (The Hunger Games, #1)",['Suzanne Collins'],4.34,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,"['young-adult', 'fiction', 'fantasy', 'science...",374.0,2008.0
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"['J.K. Rowling', 'Mary GrandPré']",4.44,Harry Potter's life is miserable. His parents ...,"['fantasy', 'fiction', 'young-adult', 'classics']",309.0,1997.0
2,3,"Twilight (Twilight, #1)",['Stephenie Meyer'],3.57,About three things I was absolutely positive. ...,"['young-adult', 'fantasy', 'romance', 'fiction...",501.0,2005.0
3,4,To Kill a Mockingbird,['Harper Lee'],4.25,The unforgettable novel of a childhood in a sl...,"['classics', 'fiction', 'historical-fiction', ...",324.0,1960.0
4,5,The Great Gatsby,['F. Scott Fitzgerald'],3.89,Alternate Cover Edition ISBN: 0743273567 (ISBN...,"['classics', 'fiction', 'historical-fiction', ...",200.0,1925.0


In [ ]:
import re
import ast
import pandas as pd

def clean_description(description):
    if pd.isna(description):
        return description

    description = re.sub(r'["""]', '', description)
    description = re.sub(r'\.\s+\.\s+\.(\s+\.)*', '...', description)
    description = re.sub(r'([.!?,;:])(?![.\s]|$)', r'\1 ', description)
    return description

def clean_authors(authors):
    if pd.isna(authors):
        return authors

    if isinstance(authors, list):
        return authors[0] if len(authors) > 0 else None
    try:
        authors_list = ast.literal_eval(authors)
        if isinstance(authors_list, list):
            return authors_list[0] if len(authors_list) > 0 else None
    except (ValueError, SyntaxError):
        pass
    return authors

def clean_title(title):
    if pd.isna(title):
        return title

    title = re.sub(r'\s?\(.*?\)', '', title)
    return title

cleaned_books_df.loc[:, 'title'] = cleaned_books_df['title'].apply(clean_title)
cleaned_books_df.loc[:, 'description'] = cleaned_books_df['description'].apply(clean_description)
cleaned_books_df.loc[:, 'authors'] = cleaned_books_df['authors'].apply(clean_authors)

cleaned_books_df['pages'] = cleaned_books_df['pages'].fillna(0).astype(int)
cleaned_books_df['original_publication_year'] = cleaned_books_df['original_publication_year'].fillna(0).astype(int)

cleaned_books_df.head()


/var/folders/fz/trg9qk8n60n_4tf1327xgb4r0000gn/T/ipykernel_27317/392036549.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_books_df['pages'] = cleaned_books_df['pages'].fillna(0).astype(int)
/var/folders/fz/trg9qk8n60n_4tf1327xgb4r0000gn/T/ipykernel_27317/392036549.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_books_df['original_publication_year'] = cleaned_books_df['original_publication_year'].fillna(0).astype(int)


,book_id,title,authors,average_rating,description,genres,pages,original_publication_year
0,1,The Hunger Games,Suzanne Collins,4.34,WINNING MEANS FAME AND FORTUNE. LOSING MEANS C...,"['young-adult', 'fiction', 'fantasy', 'science...",374,2008
1,2,Harry Potter and the Sorcerer's Stone,J.K. Rowling,4.44,Harry Potter's life is miserable. His parents ...,"['fantasy', 'fiction', 'young-adult', 'classics']",309,1997
2,3,Twilight,Stephenie Meyer,3.57,About three things I was absolutely positive. ...,"['young-adult', 'fantasy', 'romance', 'fiction...",501,2005
3,4,To Kill a Mockingbird,Harper Lee,4.25,The unforgettable novel of a childhood in a sl...,"['classics', 'fiction', 'historical-fiction', ...",324,1960
4,5,The Great Gatsby,F. Scott Fitzgerald,3.89,Alternate Cover Edition ISBN: 0743273567 (ISBN...,"['classics', 'fiction', 'historical-fiction', ...",200,1925


In [43]:
cleaned_books_df.to_csv('../data/processed/sql_data.csv', index=False)
sql_data_df = pd.read_csv('../data/processed/sql_data.csv')